# Цель работы
Разработать автоматизированное решение для корректного распознавания номеров припаркованных автомобилей на изображениях и видео, а также вычислить CER (Character Error Rate) для оценки качества.


## Установка зависимостей

In [20]:
!pip install torch torchvision seaborn pillow opencv-python easyocr numpy matplotlib nltk pandas requests

  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)


## Модуль детекции номерного знака (YOLOv5)

In [21]:
import torch
from PIL import Image

class YoloInference:
    """Сверточная модель YOLOv5 для поиска автомобильных номеров."""
    def __init__(self, weight_path: str, device=None):
        if device is None:
            device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
        self.device = device

        # загрузка кастомных весов
        self.model = torch.hub.load('ultralytics/yolov5', 'custom', path=weight_path)
        self.model.eval().to(self.device)

    def __call__(self, img: Image):
        """Возвращает список bbox‑ов [x1, y1, x2, y2]"""
        results = self.model(img)
        bboxs = results.xyxy[0][:, :4].tolist()
        return bboxs


## Обёртка над EasyOCR

In [22]:
import re
import easyocr
import numpy as np
import torch

class EasyOCRInference:
    """Минимальная обёртка над EasyOCR с allowlist рус/лат символов."""
    def __init__(self, model_params: dict, device=None):
        if device is None:
            device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
        self.device = device
        self.ocr = easyocr.Reader(**model_params, gpu=device.type == 'cuda')

    def __call__(self, img: np.ndarray):
        preds = self.ocr.recognize(img, detail=0, allowlist='0123456789ABEKMHOPCTYX')
        if preds:
            return re.sub(' ', '', preds[0])
        return None


## Spatial Transformer Network (STN) для нормализации номерного знака

In [23]:
import cv2
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from PIL import Image

def stn_preprocess(img: np.ndarray) -> torch.Tensor:
    """Препроцессинг для STN (масштабирование и нормализация)."""
    im = cv2.resize(img, (94, 24), interpolation=cv2.INTER_CUBIC)
    im = (np.transpose(np.float32(im), (2, 0, 1)) - 127.5) * 0.0078125
    return torch.from_numpy(im).float().unsqueeze(0)

def convert_image(inp: torch.Tensor) -> np.ndarray:
    """Обратное преобразование тензора в RGB‑изображение."""
    img = inp.numpy().transpose((1, 2, 0))
    img = 127.5 + img / 0.0078125
    return img.astype('uint8')[:, :, ::-1]

class STNet(nn.Module):
    # … (полный код STNet без изменений)
    def __init__(self):
        super().__init__()
        self.localization = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3),
            nn.MaxPool2d(2, 2),
            nn.ReLU(True),
            nn.Conv2d(32, 32, kernel_size=5),
            nn.MaxPool2d(3, 3),
            nn.ReLU(True),
        )
        self.fc_loc = nn.Sequential(
            nn.Linear(32 * 14 * 2, 32),
            nn.ReLU(True),
            nn.Linear(32, 6),
        )
        # Инициализация аффинного преобразования как единичного
        self.fc_loc[2].weight.data.zero_()
        self.fc_loc[2].bias.data.copy_(torch.tensor([1, 0, 0, 0, 1, 0], dtype=torch.float))

    def forward(self, x):
        xs = self.localization(x)
        xs = xs.view(-1, 32 * 14 * 2)
        theta = self.fc_loc(xs).view(-1, 2, 3)
        grid = F.affine_grid(theta, x.size(), align_corners=False)
        return F.grid_sample(x, grid, align_corners=False)

class STNetInference:
    """Обёртка над STN для вызова как функции."""
    def __init__(self, weight_path: str, device=None):
        device = device or torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
        self.device = device
        self.stn = STNet().to(device)
        self.stn.load_state_dict(torch.load(weight_path, map_location=device))
        self.stn.eval()

    def __call__(self, image: np.ndarray) -> Image:
        bgr = image[:, :, ::-1].copy()
        data = stn_preprocess(bgr).to(self.device)
        with torch.no_grad():
            transformed = self.stn(data).cpu()
        grid_img = torchvision.utils.make_grid(transformed)
        rgb = convert_image(grid_img)
        return Image.fromarray(rgb)


## Универсальные функции решения
Функции соответствуют требованиям задания: `preprocess`, `get_plate`, `get_clean_plate`, `get_raw_text`, `get_number`.

In [24]:
import time
import cv2
import numpy as np
from PIL import Image

# --- 1. Предобработка входного изображения ---
def preprocess(img: Image) -> Image:
    """Приводим изображение к RGB‑формату без изменений размеров."""
    return img.convert('RGB')

# --- 2. Поиск номерного знака ---
def get_plate(preprocessed_img: Image, detector) -> list[Image]:
    """Возвращает список вырезанных знаков (может быть несколько)."""
    bboxes = detector(preprocessed_img)
    plates = []
    for x1, y1, x2, y2 in bboxes:
        plates.append(preprocessed_img.crop((int(x1), int(y1), int(x2), int(y2))))
    return plates

# --- 3. Очистка / выравнивание ---
def get_clean_plate(plate: Image, transformer=None) -> Image:
    """STN‑нормализация (если есть), иначе исходное изображение."""
    if transformer is not None:
        return transformer(np.asarray(plate))
    return plate

# --- 4. OCR ---
def get_raw_text(clean_plate: Image, ocr) -> str | None:
    """Распознаёт текст на нормализованном изображении."""
    return ocr(np.asarray(clean_plate))

# --- 5. Пост‑обработка текста ---
import re

def get_number(raw_text: str | None) -> str | None:
    """Приведение строки к шаблону `БЦЦЦББ РРР` или None."""
    if raw_text is None:
        return None
    pattern = r'([ABEKMHOPCTYX]{1}\d{3}[ABEKMHOPCTYX]{2})(\d{3})'
    m = re.match(pattern, raw_text)
    if m:
        return f"{m.group(1)} {m.group(2)}"
    return None


## Загрузка весов и создание экземпляров моделей

In [25]:
YOLO_PATH = './weights/detection/yolov5.pt'
STN_PATH = './weights/transform/stn.pth'
OCR_CONFIG = {
    'lang_list': ['en'],
    'recog_network': 'easyocr_custom',
    'user_network_directory': './ocr/',
    'model_storage_directory': './weights/ocr/',
    'detector': False,
}

detect_model = YoloInference(weight_path=YOLO_PATH)
ocr_model = EasyOCRInference(model_params=OCR_CONFIG)
transform_model = STNetInference(weight_path=STN_PATH)


Using cache found in C:\Users\user/.cache\torch\hub\ultralytics_yolov5_master


requirements: Ultralytics requirements ['gitpython>=3.1.30', 'setuptools>=70.0.0'] not found, attempting AutoUpdate...
   ---------------------------------------- 1.2/1.2 MB 4.3 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 65.5.0
    Uninstalling setuptools-65.5.0:-------- 1/4 [setuptools]
      Successfully uninstalled setuptools-65.5.0[setuptools]
   ---------------------------------------- 4/4 [gitpython]]

requirements: AutoUpdate success  20.9s
WARNING requirements: Restart runtime or rerun command for updates to take effect



YOLOv5  2025-4-1 Python-3.11.9 torch-2.7.1+cpu CPU

Fusing layers... 
custom_YOLOv5s summary: 157 layers, 7012822 parameters, 0 gradients, 15.8 GFLOPs
Adding AutoShape... 
Using CPU. Note: This module is much faster with a GPU.


## Обработка входных изображений

In [26]:
def process_image(path: str) -> str | None:
    """Полный конвейер получения номера из файла изображения."""
    img = Image.open(path)
    img = preprocess(img)
    plates = get_plate(img, detector=detect_model)
    for plate in plates:
        clean = get_clean_plate(plate, transformer=transform_model)
        raw = get_raw_text(clean, ocr=ocr_model)
        number = get_number(raw)
        if number:
            return number
    return None
path = './'
test_imgs = [path+'/data/car1.jpg', path+'/data/car2.jpg', path+'/data/car3.jpg']
predicted_numbers = [process_image(p) for p in test_imgs]
expected_numbers = ['H764KE 799', 'C005KM 190', 'A715AE 977']
for fname, pred in zip(test_imgs, predicted_numbers):
    print(f'{fname}: {pred}')


C:\Users\user/.cache\torch\hub\ultralytics_yolov5_master\models\common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
C:\Users\user/.cache\torch\hub\ultralytics_yolov5_master\models\common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
C:\Users\user/.cache\torch\hub\ultralytics_yolov5_master\models\common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):


.//data/car1.jpg: H764KE 799
.//data/car2.jpg: C005KM 190
.//data/car3.jpg: A715AE 977


## Оценка качества (CER)

In [27]:
import nltk
import numpy as np
from nltk.metrics.distance import edit_distance

# ------------------------------------------------------------------------
# 1. Функция CER для одной пары строк
# ------------------------------------------------------------------------
def cer(predicted: str, ground_truth: str) -> float:
    """
    Вычисляет Character Error Rate (CER) по формуле:
        CER = edit_distance(predicted, ground_truth) / len(ground_truth)

    :param predicted: Распознанная строка.
    :param ground_truth: Эталонная строка.
    :return: Доля ошибочных символов (0.0 — идеально, 1.0 — полный разнобой).
    :raises ValueError: Если ground_truth пуст.
    """
    if len(ground_truth) == 0:
        raise ValueError("Ground-truth string must not be empty.")

    # NLTK возвращает минимальное число операций (вставка, удаление, замена)
    edits = edit_distance(predicted, ground_truth)
    return edits / len(ground_truth)


# ------------------------------------------------------------------------
# 2. Пакетный расчёт CER по спискам
# ------------------------------------------------------------------------
def batch_cer(predictions, references):
    """
    Вычисляет CER для каждой пары и возвращает массив NumPy с результатами.

    :param predictions: Список (или иной итерируемый объект) предсказанных строк.
    :param references:  Список эталонных строк той же длины.
    :return: np.ndarray с индивидуальными значениями CER.
    :raises ValueError: Если размеры входных списков не совпадают.
    """
    if len(predictions) != len(references):
        raise ValueError("Number of predictions and references must be equal.")

    char_error_rates = []
    for pred, ref in zip(predictions, references):
        char_error_rates.append(cer(pred, ref))
    return np.array(char_error_rates, dtype=float)

# Расчёт CER для всех пар
cer_values = batch_cer(predicted_numbers, expected_numbers)

# ➜ Вывод индивидуальных метрик
for gt, pred, c in zip(expected_numbers, predicted_numbers, cer_values):
    print(f"Реальный:{gt}\nПрогноз: {pred}\nCER:  {c:.3f}\n")

# ➜ Среднее значение по выборке
print(f"Средний CER: {cer_values.mean():.3f}")


Реальный:H764KE 799
Прогноз: H764KE 799
CER:  0.000

Реальный:C005KM 190
Прогноз: C005KM 190
CER:  0.000

Реальный:A715AE 977
Прогноз: A715AE 977
CER:  0.000

Средний CER: 0.000


## 8. (Опция) Обработка видео `cam1.mp4`

In [28]:
from __future__ import annotations

import cv2
from pathlib import Path
from typing import List

from PIL import Image

# --------------------------------------------------------------------------- #
# вспомогательная обёртка: полный конвейер «картинка ➜ номер» из ноутбука     #
# --------------------------------------------------------------------------- #
def recognize_license_plate(img: Image.Image) -> str | None:
    """
    Возвращает распознанный номер в формате «БЦЦЦББ РРР» либо *None*.

    Здесь используется уже обученный пайп-лайн, содержащий функции:
    ``preprocess → get_plate → get_clean_plate → get_raw_text → get_number``.
    Все они должны быть объявлены ранее в ноутбуке (см. основное решение).
    """
    img = preprocess(img)                                     # 1. базовая предобработка
    for plate in get_plate(img, detector=detect_model):       # 2. детекция bbox-ов
        clean = get_clean_plate(plate, transformer=transform_model)  # 3. STN-выравнивание
        raw = get_raw_text(clean, ocr=ocr_model)              # 4. OCR (EasyOCR)
        num = get_number(raw)                                 # 5. пост-обработка
        if num:
            return num
    return None


# --------------------------------------------------------------------------- #
# основная функция: выборка равномерно распределённых *max_keyframes* кадров  #
# --------------------------------------------------------------------------- #
def extract_plate_numbers_from_video(                          
    video_path: str | Path,
    max_keyframes: int = 10,
) -> List[str]:
    """
    Выбирает до *max_keyframes* равномерно распределённых кадров из видео и
    извлекает уникальные номерные знаки.

    Parameters
    ----------
    video_path : str | pathlib.Path
        Путь к видеоролику (любой кодек, поддерживаемый OpenCV).
    max_keyframes : int, default=10
        Максимальное число кадров, которые будут обработаны.

    Returns
    -------
    List[str]
        Уникальные номера в порядке появления.
    """
    # --- инициализация видеопотока -------------------------------------------------
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise IOError(f'Не удаётся открыть файл: {video_path}')
    
    def canonical(s: str) -> str:    # нормализуем «H764KE 799» → «H764KE799»
        return s.replace(' ', '').upper()

    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if frame_count <= 0:
        cap.release()
        raise ValueError('Не удалось определить количество кадров в видео.')

    step = max(frame_count // max_keyframes, 1)  # интервал выборки
    collected: List[str] = []                    # итоговый список номеров
    seen: set[str] = set()           # быстрая проверка дубликатов

    idx = 0
    # --- основной цикл -------------------------------------------------------------
    while cap.isOpened() and len(collected) < max_keyframes:
        ret, frame_bgr = cap.read()
        if not ret:
            break

        # выбираем каждый *step*-ый кадр
        if idx % step == 0:
            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
            pil_img = Image.fromarray(frame_rgb)

            plate_num = recognize_license_plate(pil_img)
            
            if plate_num and len(plate_num) in (9, 10) and plate_num not in collected:
                key = canonical(plate_num)
                if key not in seen:                           # добавление только новых знаков без дублей
                    seen.add(key)
                    collected.append(plate_num)

        idx += 1

    cap.release()
    return collected

video_numbers = extract_plate_numbers_from_video(path+'/data/cam1.mp4', max_keyframes=4)
print(" ".join(video_numbers))


C:\Users\user/.cache\torch\hub\ultralytics_yolov5_master\models\common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
C:\Users\user/.cache\torch\hub\ultralytics_yolov5_master\models\common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
C:\Users\user/.cache\torch\hub\ultralytics_yolov5_master\models\common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
C:\Users\user/.cache\torch\hub\ultralytics_yolov5_master\models\common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
C:\Users\user/.cache\torch\hub\ultralytics_yolov5_master\models\comm

H764KE 799 C005KM 190 A715AE 977


## Выводы
В ходе работы разработан полноценный конвейер распознавания автомобильных номеров, включающий детекцию, аффинное выравнивание, OCR и пост‑обработку. Средний CER показывает высокое качество решения.